# Learning Notes

Personal log of what I've learned working on this project: concepts, tools, gotchas, things I want to remember.

## What is YAML

YAML ("YAML Ain't Markup Language") is a human-readable format for structured data, mostly used for config files. It uses indentation instead of brackets/braces, `key: value` pairs, and `- ` for list items.

The `data.yaml` file the Roboflow dataset download produced is a real example:

```yaml
path: C:/Users/rohan/Desktop/Quant Sports Project/experiments/Football-Players-Detection-1
train: train/images
val: valid/images
test: test/images

nc: 4
names: ['ball', 'goalkeeper', 'player', 'referee']
```

- `path`, `train`, `val`, `test`: tell YOLO where to find the dataset's image folders.
- `nc`: number of classes (4 here).
- `names`: the class labels, in the same order the model will predict them.

Nesting works by indentation, for example the `roboflow:` block underneath that has its own `workspace`, `project`, `version`, etc. as sub-keys. No special syntax needed, just consistent indentation.


## COCO weights, and why fine-tuning beats starting from scratch

COCO (Common Objects in Context) is a large general-purpose dataset, about 330,000 images across 80 everyday object classes (person, car, dog, sports ball, tv, etc.). "COCO-pretrained weights" means the model has already been trained on that dataset, so it already knows general visual concepts: edges, shapes, textures, what a person-shaped blob looks like from different angles, and so on.

That's exactly what `yolo26n.pt` was in the first smoke test: a COCO-pretrained checkpoint, used purely for inference (no training). It worked reasonably for the generic `person` class but was unreliable for `sports ball` and had no concept of soccer-specific roles (player vs. referee vs. goalkeeper), because COCO never taught it those distinctions.

**Fine-tuning** (what the Roboflow retrain decision is doing) means continuing training from those COCO-pretrained weights on a new, task-specific dataset, instead of starting from random weights. This is a form of transfer learning:

- The early layers of the network already know general visual features, so training doesn't have to relearn "what an edge looks like" from zero.
- Only the later layers really need to adapt, mainly the classification head, to the new classes (`ball`, `goalkeeper`, `player`, `referee`).
- This means far less data and far less compute is needed to get a good result, compared to training an object detector completely from scratch, which typically needs hundreds of thousands of images.

This is why the plan is "fine-tune YOLO26 on the Roboflow dataset" rather than "train a brand new detector from nothing."


## Why 4GB VRAM is little, and what CUDA actually does

**VRAM** is the memory that lives on the GPU itself (separate from regular system RAM). During training, VRAM has to hold: the model's weights, the batch of images currently being processed, all the intermediate activations from the forward pass, and the gradients from the backward pass, all at once. All of that competes for the same pool of memory.

4GB is small by current standards. The GTX 1650 in this machine is a budget/laptop-class GPU; modern training GPUs (RTX 4090, A100, H100) have 24GB to 80GB+. The practical consequence: if the batch size or image size is too large for the available VRAM, training crashes with an out-of-memory (OOM) error rather than just running slower. That's why the training run started with `batch=16` instead of YOLO's larger default, as a conservative starting point, with room to raise it if VRAM allows or lower it if it OOMs.

**CUDA** is NVIDIA's platform that lets software run computations directly on the GPU instead of the CPU. The reason this matters for deep learning: a CPU has a small number of powerful, general-purpose cores, while a GPU has thousands of simpler cores built to do the same operation on lots of data at once. Deep learning is mostly large matrix multiplications, which is exactly the kind of highly-parallel, repetitive math GPUs are built for. CUDA is the layer that lets PyTorch hand those matrix operations off to the GPU's cores instead of the CPU's.

Concretely, in this project: `torch` was first installed as a CPU-only build (`2.14.0+cpu`), so `torch.cuda.is_available()` returned `False` and training would have run on the CPU, slowly. Reinstalling `torch`/`torchvision` from the CUDA-enabled index (`cu130`, matching this machine's driver) fixed that; `torch.cuda.is_available()` now returns `True` and the GTX 1650 is detected and usable for training.


## Lesson (2026-09-18): local GPU training on old/underpowered hardware is a real hardware risk, not just slow

While attempting to run training locally on the GTX 1650, a capacitor on the machine blew, likely a short circuit related to the power adapter under sustained load. Fixed, no lasting damage, but the practical lesson is:

Training deep learning models pushes a machine's power delivery and cooling much harder than normal use, for a sustained period (not a quick spike). On older or already-marginal hardware, that sustained load is a genuine risk to the hardware itself, not just something that runs slowly.

**Decision going forward:** use an external/cloud GPU (Google Colab, or similar) for actual training runs instead of pushing local hardware. Local setup is still fine for quick inference smoke tests (like the original YOLO detection test), just not for sustained training workloads on this machine.


## Workflow (2026-09-20): shifting training from local to Google Colab

Following the local GPU hardware fault (previous entry), the actual fine-tuning run moves to Google Colab instead of this machine. Two ways to drive Colab, and why one was picked over the other:

**Option A: the official `google-colab-cli`.** Lets you provision a GPU (`colab new -s mysession --gpu T4`), install deps, and run a plain `.py` script unattended (`colab exec -f train.py`) entirely from a terminal, no browser tab needed. Built for headless/automated runs. Problem: it only supports Linux and macOS, not Windows, so on this machine it would need WSL2 first.

**Option B: upload a notebook to colab.research.google.com and run it interactively.** No install, no WSL, works from any browser. This is the one actually used, given the Windows constraint. Practical differences from running the same code locally:
- Colab's disk is remote and ephemeral (wiped when the session ends), so local file paths (like the smoke test's hardcoded `C:\Users\rohan\...\08fd33_4.mp4`) mean nothing there. Anything the training code reads either has to be re-downloaded inside Colab (the Roboflow dataset, via its API) or mounted from Google Drive.
- Secrets can't come from a local `.env` file, since that file never leaves this machine. Colab has its own per-notebook Secrets panel instead (see the next entry).
- Trained weights need to be pulled back out before the session ends: either download the file directly, or mount Drive and save there during training.

**Where the code lives:** the training notebook is `pipeline/detection/train.ipynb`, not `experiments/`. `experiments/` is reserved (per `decision_log.md`, 2026-09-18) for diagnostic spikes like the pretrained YOLO smoke test. This training run is the actual Stage 1 pipeline deliverable the smoke test's findings led to, not a spike, so it gets its own real location, one that mirrors CLAUDE.md's numbered pipeline stages (`pipeline/detection/` = Stage 1, `pipeline/calibration/` would be Stage 2, and so on).


## Lesson (2026-09-20): why `.env` (or any secret) must never be committed to git

`.env` holds the Roboflow API key as a plain `KEY=value` line, read in Python via `python-dotenv`'s `load_dotenv()` plus `os.environ["ROBOFLOW_API_KEY"]`. It's listed first in `.gitignore` for a reason: if a secret ever gets committed, deleting it in a later commit does **not** remove it. Git keeps full history, so the key still sits in the repo's `.git` history, retrievable by anyone with access to the repo, forever, unless the history itself is rewritten (a destructive operation, and not a reliable fix once something's been pushed anywhere public). On a public repo, bots actively scan new commits for exposed API keys within minutes, so a leaked key there should be treated as immediately compromised, not just "risky."

The same risk shows up in a different shape on Colab. There's no `.env` file there (see previous entry), but pasting the raw key directly into a notebook code cell has the identical problem: if that notebook is ever saved and pushed to the repo with the key still typed into a cell, the key is committed exactly as if it were in a tracked `.env` file. That's why the Colab version uses `google.colab.userdata.get("ROBOFLOW_API_KEY")` instead: the key lives in Colab's Secrets panel (tied to the Google account, not the notebook file), and `userdata.get(...)` only pulls the value in at runtime. Nothing about the actual key ever gets written into the `.ipynb` file's saved source.

General rule this reinforces: secrets flow through environment variables or a dedicated secret store, read at runtime, never typed as a literal value into any file that gets committed.


## Concept (2026-09-20): reading YOLO training output, loss terms and mAP

Every epoch, `yolo mode=train` prints a row with three loss values and, after each epoch, a validation row with precision, recall, and two mAP scores. They measure different things and get watched differently.

**The three losses (should trend down over epochs):**
- `box_loss`: how far off the predicted bounding box's position and size are from the ground-truth box (a CIoU-style loss: penalizes overlap, center distance, and aspect ratio mismatch together).
- `cls_loss`: how confidently and correctly the model predicts the right class for each detected box (`ball`, `goalkeeper`, `player`, `referee` in this dataset). High `cls_loss` means it's unsure or picking the wrong class, not that boxes are misplaced.
- `l1_loss`: a simple L1 penalty on the raw box coordinate error. It is tiny in magnitude (around 0.001 to 0.002 in the first run), so it barely moves compared to the other two.

**Correction (2026-09-20):** an earlier version of this entry listed `dfl_loss` (Distribution Focal Loss) as the third loss. That is what YOLOv8 and YOLO11 report, but YOLO26 removed DFL from its detection head and replaced it with direct box regression plus L1 loss, which is why the real training log shows `l1_loss` instead. Lesson: the loss columns depend on the model version, so check the actual log rather than assuming from older YOLO docs.

All three are *training-set* losses: what the optimizer is directly minimizing during the backward pass. Going down means the model fits the training data better; it does not by itself mean the model generalizes.

**`mAP50` and `mAP50-95` (should trend up, evaluated on the validation split after every epoch):**
- A predicted box counts as a correct detection if it overlaps the ground-truth box by at least some IoU (Intersection over Union) threshold. `mAP50` requires 50% overlap; `mAP50-95` averages accuracy across ten thresholds from 50% to 95% in 5% steps, a stricter, more comprehensive score (the same metric standard in the COCO benchmark).
- "Average Precision" itself is the area under a class's precision-recall curve; "mean" AP averages that across all classes (ball, goalkeeper, player, referee here), so one weak class quietly pulls the whole number down even if the others are strong.

**Why watch both, not just the losses:** loss is computed on training data and always has some incentive to keep dropping; mAP is computed on held-out validation data and is the actual measure of whether the model works on data it wasn't trained on. If losses keep falling while `mAP50-95` plateaus or drops, that's the classic overfitting signal, worth stopping the run for rather than letting all 100 epochs finish. A `nan` appearing in any loss column is a different failure mode entirely (usually a learning-rate or bad-data problem), and also worth stopping for immediately rather than waiting it out.


## Concept (2026-09-20): the `runs/` folder, and why it must be gitignored

Every time Ultralytics runs a `train`, `val`, or `predict` command, it automatically creates a `runs/` folder next to wherever the command was run from, and saves that run's output in a numbered subfolder (for example `runs/detect/train/`, then `train2/`, `train3/` on later runs). Nothing has to be configured for this, which is why a `runs/` folder appeared without anyone creating it.

**What ends up inside:**
- `weights/best.pt` and `weights/last.pt`: the trained checkpoints.
- `results.csv` and `results.png`: the per-epoch losses and mAP, as numbers and as curves.
- Confusion matrix and precision-recall plots.
- Annotated images: sample training batches and validation predictions with boxes drawn on them. For `predict` runs on video, the output is the annotated video itself.

**Why it needs a `.gitignore` entry:**
- **Derived frames.** The annotated images and videos are derived from footage. CLAUDE.md's rule is that public repo output may contain code and aggregated statistics only, never raw video or derived frames, because of DFL and SoccerNet redistribution restrictions. Committing `runs/` by accident would break that rule.
- **Size and reproducibility.** The contents are large and can be regenerated by re-running the command, so they add weight to the repo without adding anything that needs version history.

**How the pattern works:** the `.gitignore` already had `experiments/runs/`, but that only matches that one exact path. A run from `pipeline/detection/` creates `pipeline/detection/runs/`, which the old line would miss. A pattern with no folder prefix, just `runs/`, matches a folder with that name at any depth in the repo. The trailing slash means it only matches directories.

**Gotcha:** `.gitignore` only affects files git is not already tracking. If a file was committed before its ignore rule existed, it stays tracked, and it has to be removed from git explicitly (`git rm --cached`). Adding the rule early, before any run output exists, is the safe order. To check that a path is ignored, run `git check-ignore -v <path>`, which prints the rule that matched.


## Concept (2026-09-21): `YOLO("file.pt")` is how you pick which weights a model uses

```python
from ultralytics import YOLO

model = YOLO("football_yolo26n_best.pt")
results = model.predict("clip.mp4", save=True)
```

The string passed to `YOLO(...)` is the path to a weights file (a `.pt` checkpoint). That one argument decides which trained model you get. Nothing else in the code changes:
- `YOLO("yolo26n.pt")` loads the stock COCO model (80 classes: person, sports ball, tv, ...).
- `YOLO("football_yolo26n_best.pt")` loads the model fine-tuned on Colab (4 classes: ball, goalkeeper, player, referee).

**This is how off-site training gets used locally.** Training ran on Colab's GPU and produced `best.pt` there. Downloading that file and passing its path to `YOLO(...)` on this machine is the whole hand-off. The `.pt` file bundles the network architecture, the learned weights, and the class names, so the local code does not need to know anything about how or where it was trained. Loading it is the same step whether the weights came from Colab, from a local run, or from Ultralytics' pretrained downloads.

**Checking which model you actually loaded:** `model.names` lists the classes it can predict. The COCO model has 80 entries and the fine-tuned one has 4, which is how the two `.pt` files in `pipeline/detection/` were told apart.

**Path gotcha:** a relative path like `"football_yolo26n_best.pt"` is resolved from the notebook's own folder (`pipeline/detection/`), not the project root. If the path is wrong, the notebook fails with a file-not-found error. The exception is official Ultralytics names such as `yolo26n.pt`: if that file is missing, Ultralytics silently downloads it into the current folder, which is why a stray copy of the COCO checkpoint appeared in `pipeline/detection/`.

The command-line version of the same thing is the `model=` argument, for example `yolo task=detect mode=val model=football_yolo26n_best.pt ...`.


## Concept (2026-09-21): what OpenCV (`cv2`) is, and where it sits in the pipeline

`cv2` is OpenCV, a general-purpose library for images and video. It is installed as `opencv-python` but imported as `cv2` (a naming leftover from the library's history). Ultralytics already depends on it, which is why it was in the `.venv` without being installed separately.

**Split to remember:** YOLO (Ultralytics, running on PyTorch) decides *what is where* in a frame. OpenCV handles *pixels, video, and geometry*.

**Where cv2 shows up in the pipeline:**
- **Before detection:** `cv2.VideoCapture` reads frames, `cv2.VideoWriter` saves them.
- **After detection, presentation:** drawing rings, triangles, and labels (`cv2.ellipse`, `cv2.drawContours`, `cv2.putText`).
- **After detection, data:** this is not presentation. `cv2.findHomography` and `cv2.warpPerspective` convert pixel positions into real pitch coordinates (Stage 2, calibration), and every speed or distance figure depends on it. `cv2.kmeans` on jersey pixels can assign teams (Stage 3). These steps produce numbers, not pictures.
- **Tracking itself** (ByteTrack and similar) mostly works from the detection boxes with motion math, not from OpenCV.

Pipeline order: `read frames (cv2) -> detect (YOLO) -> track (tracker) -> calibrate and assign teams (cv2) -> features -> draw (cv2, optional)`.

For live trading, drawing is the only optional step. It is useful for debugging, but trading logic runs on coordinates and features. Calibration is not optional.


## Concept (2026-09-21): drawing on frames with cv2, and why it must work one frame at a time

Code lives in `pipeline/common/drawing.py` (`draw_ellipse`, `draw_triangle`, `annotate_frame`). Shared helpers go in `pipeline/common/`, not in one stage's folder.

**Coordinates.** A frame is a NumPy array of shape `(height, width, 3)`. The origin `(0, 0)` is the top-left, x grows right, and y grows *down*. So the bottom of a box is its larger y (`y2`), and "above the ball" means a smaller y.

**Positions versus sizes.** `x1, y1, x2, y2` are positions (where something is). `width = x2 - x1` is a size (how big it is). A size comes from a difference, never from a raw position. Setting the ring's half-height to `y2 / 2` was a mistake: `y2` depends on where the player stands in the frame, so the ring would change size with position. Half-height should come from the box width instead (`b = max(1, int(0.35 * width))`).

**`cv2.ellipse(frame, center, axes, angle, startAngle, endAngle, color, thickness)`.**
- `center` is `(cx, y2)`, the bottom-center of the box, so the ring sits at the feet.
- `axes` is `(a, b)`, the half-width and half-height. The ring is squashed (`b` smaller than `a`) because the camera views the pitch at an angle.
- The start and end angles are in degrees, measured clockwise from the right side (3 o'clock), because y points down. `0` to `360` is a full ring. `-45` to `235` leaves a gap at the top.

**Gotchas hit while writing it:**
- **Colors are BGR, not RGB.** `(0, 255, 255)` is yellow. Showing a frame with matplotlib needs `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)` first, or the colors look swapped.
- **`/` gives a float, `//` gives an int.** `cx = (x1 + x2) / 2` crashed because OpenCV only accepts integer coordinates. The error message was misleading (`ellipse() takes at most 5 arguments (8 given)`); it really meant the center could not be parsed. Use `//`.
- **Return the frame.** A function ending in `pass` returns `None`, so `frame = draw_ellipse(...)` would replace the frame with `None`. OpenCV also draws directly on the array it is given, so the original frame changes; call `frame.copy()` first to keep an untouched version.

**Triangle.** Three `[x, y]` points in a NumPy array with dtype `np.int32`: the tip just above the ball's box, and two corners higher up to the left and right. `cv2.drawContours(frame, [points], 0, color, cv2.FILLED)` fills it; the points go inside a list and `0` means "the first contour". A second call with a black color and thickness 2 adds an outline so it stays visible on the pitch.

**Reading results.** `result.boxes.xyxy` is a torch tensor, so `.tolist()` gives plain numbers. `result.names[int(cls)]` turns a class number into its name (`ball`, `player`, ...). A `COLORS` dictionary with `.get(name, default)` avoids a crash on an unknown class.

**Why per frame matters (live).** The drawing functions take one frame in and return one frame out, so they work identically on a file or a live stream. The batch pattern does not: a `read_video` that returns a list of every frame needs about 6 MB per 1080p frame, roughly 4.4 GB for this 750-frame clip and hundreds of GB for a full match, and a live stream never ends. The fix is a generator (a function that uses `yield` to hand back one frame at a time), the same idea as Ultralytics' `stream=True`. Related traps in that code: a hardcoded 24 fps when the clip is 25 fps (speed and distance depend on real timing), no check that the file opened, and a missing `cap.release()`. For live, also budget time: at 25 fps there are 40 ms per frame, and detection alone took about 45 ms on CPU.

**Licensing note.** The reference repo (`abdullahtarek/football_analysis`) declares no license, so its code is all-rights-reserved by default. The drawing style (an ellipse at the feet, a triangle for the ball) was reimplemented rather than copied.

**Notebook import trick.** A notebook runs from its own folder, so to import from `pipeline/common/`, add the project root to the path (`sys.path.append("../..")`) and turn on `%load_ext autoreload` with `%autoreload 2` so edits to the `.py` file are picked up without restarting the kernel.


## Concept (2026-09-22): Frame-by-frame vs batch processing, and the latency tradeoff

**Two ways to process a video:**

**Batch processing (reference tracker approach):**
```python
frames = [load all 750 frames into memory]
detections = model.predict(frames[0:20])  # predict on 20 at once
detections += model.predict(frames[20:40])
# ... continue in batches ...
```
- Pros: GPU is more efficient (processes multiple frames at once), amortizes model setup overhead
- Cons: **must wait for the batch to fill before inference starts**, adds N×40ms latency buffer before any output

**Frame-by-frame (our Tracker approach):**
```python
while True:
    frame = cap.read()
    detection = model.predict(frame)  # predict on 1 frame
    tracks = tracker.update(detection)
    # output immediately
```
- Pros: lowest latency, output is ready after one frame's work, works on live streams (no batch boundary)
- Cons: GPU is less efficient (can't parallelize across frames), slightly slower per-frame inference due to model setup

**For live in-play trading:**
Batch processing is a non-starter. Even collecting 2-3 frames before processing (20-30ms buffer) is unacceptable when the market can move in milliseconds. Frame-by-frame is the only viable approach.

**Why this matters for the project:**
The Tracker class was designed frame-by-frame from the start, not as an afterthought. The latency constraint (40ms per frame) drove the architecture. Later stages (calibration, features) must also be frame-by-frame for the pipeline to remain live-ready.

**One exception:** backtesting on historical data can use batch processing for speed (no latency requirement), then switch to frame-by-frame for live. This is a common pattern: development on batches for speed, production on frames for correctness.


## Concept (2026-09-22): Ball interpolation, and why it helps with weak detection

**The problem:** the fine-tuned YOLO26 detector still misses the ball in about 35% of frames (ball recall ~0.33-0.36 on test set). In a live stream, gaps in ball tracking break feature calculations (e.g., "distance to ball" becomes undefined).

**Ball interpolation (simple carry-forward approach):**
If the ball is not detected in frame N:
1. Check if the ball was detected in frame N-1
2. If yes, re-add the last known ball bounding box to frame N's detections
3. Mark it with lower confidence (0.5 vs real detections at ~0.7-0.9) so it ranks lower if NMS conflicts

**Why it works:**
- The ball moves smoothly across the pitch; in 40ms (one frame at 25fps) it doesn't jump far
- Reusing the last position is a safe assumption for one or two frames
- ByteTrack will handle the ID continuity; the same ball bbox (even interpolated) gets the same track ID

**Limitations:**
- Only handles single-frame misses; if the ball is missed for 3+ consecutive frames, position drifts
- Doesn't help if the ball leaves the visible area (e.g., goes behind a player); in that case, interpolation keeps drawing it where it last was
- Assumes the ball is always on the pitch; won't handle crowd balls or reflections

**In this project:**
Implemented as `last_ball_bbox` state tracking in the Tracker class. When ball is missing, the interpolated bbox is added with confidence 0.5. This is a "good enough" fix for development; a better approach would be a separate ball-tracking model or Kalman filter prediction, but those add complexity.

**Cost:** ~1ms per frame (just appending to the tracks array and setting state).

**Update (2026-09-25):** corrections to the entry above.
- "ByteTrack will handle the ID continuity" was never true. The carried-forward ball was appended *after* ByteTrack ran, so it never got a ByteTrack ID. It had a hardcoded ID of 1, which collided with a player, and class 2 (player) instead of ball. Both are fixed: since 2026-09-25 the ball skips ByteTrack entirely and always has ID -1.
- It is not limited to single-frame misses: it carries the ball forward for up to `max_ball_miss` frames (default 5), then drops it. Before that limit existed, a ball that left the frame stayed frozen at its last spot forever.
- "~1ms" was an estimate, never measured.
- Carry-forward freezes the ball in place. The planned replacement is a Kalman filter, which predicts where the ball moves and how uncertain that guess is.


## Concept (2026-09-22): Non-Maximum Suppression (NMS), and how it solves double-counting

**The problem:** after YOLO detection, sometimes the same object is detected twice in one frame — two overlapping boxes on the same person. This creates duplicate track IDs and confuses downstream processing.

**NMS (Non-Maximum Suppression) is a post-processing step that removes redundant boxes:**
1. Sort all boxes by confidence score (high to low)
2. Keep the highest-confidence box
3. Calculate Intersection over Union (IoU) with all remaining boxes
4. Remove any box with IoU > threshold (default 0.3 here, meaning >30% overlap)
5. Repeat with the next highest-confidence box

**Example:** if two boxes both detect the same player, and they overlap 40% (IoU > 0.3), the lower-confidence box is discarded, keeping only the higher-confidence one.

**Why threshold matters:**
- threshold=0.5: conservative, allows 49% overlap (misses some duplicates)
- threshold=0.3: aggressive, removes anything >30% overlap (catches more duplicates)
- threshold=0.1: very aggressive, removes almost all overlaps (but risks removing legitimate nearby objects)

**In this project:**
NMS reduced double-counting but didn't eliminate it entirely, tested at threshold 0.5 and 0.3. Remaining duplicates are likely boxes that don't overlap enough to trigger NMS, meaning the detector found two genuinely separated parts of the same object (e.g., two corners of a player's bounding box detected separately). This is accepted as a minor limitation; fixing it would require detector retraining or more sophisticated deduplication.

**Cost:** NMS adds ~2-3ms overhead per frame (comparing overlaps is fast but not free). In latency-critical systems, sometimes NMS is skipped if double-counting is rare.

**Update (2026-09-25):** corrections to the entry above.
- The check that "NMS reduced double-counting" was done on a video that drew raw YOLO detections (before NMS and ByteTrack), so it never actually tested NMS. It needs redoing on tracker output.
- By default `with_nms()` only compares boxes of the same class. One person detected as both `player` and `referee` keeps both boxes at any threshold. That may be the real cause of the duplicates that "didn't overlap enough". `class_agnostic=True` compares across classes; testing it is Track 1 step 5.
- Since 2026-09-25, NMS runs on people only; the ball skips it.
- "~2-3ms" is an estimate, not measured.


## Concept (2026-09-22): GPU vs CPU latency for inference, and why it matters for live trading

**Latency measurements on the same YOLO26 detector:**
- CPU (this machine): ~58ms mean per frame
- GPU (T4 on Colab, estimated): ~15-20ms mean per frame

**Why the difference:**
A GPU has thousands of simple cores designed to do the same operation on many data points in parallel. Deep learning is billions of matrix multiplications, which is exactly that pattern. A CPU has only 8-16 powerful cores that do general-purpose work sequentially. Inference (forward pass) is read-only math with no backward pass, so a GPU doesn't need the 4GB+ VRAM that training needs, but it still crushes a CPU for sheer throughput.

**For live in-play trading:**
- 25 fps match = 40ms budget per frame
- CPU at 58ms is already over budget, before calibration (~10-20ms) and feature extraction (~5-10ms) are added
- Total CPU pipeline: ~80-100ms per frame → too slow for competitive order placement
- Total GPU pipeline: ~30-40ms per frame → feasible for live trading

**Trade-off:**
GPU adds infrastructure (Colab or local RTX), but it's essential for live. CPU is fine for development/validation. The project's architecture (frame-by-frame processing) is GPU-agnostic; only the latency constraint changes.

**Practical note:** Colab's free tier has limited GPU hours; paid Colab or a local GPU (RTX 4060+) is needed for production. Cost vs. latency is a real tradeoff, not a free lunch.


## Concept (2026-09-25): ByteTrack's hidden filter, and why it was dropping the ball

**How ByteTrack decides what to track.** ByteTrack does not look at pixels. Each frame it gets a list of boxes with confidences and links them to the tracks it already has by box overlap (IoU). Three rules, from the `supervision` 0.30.4 source (`sv.ByteTrack()` defaults):
- A box can only **start a new track** if its confidence is at least 0.35 (`track_activation_threshold` 0.25, plus 0.1 added inside the code).
- A box between 0.1 and 0.25 can only **extend a track that already exists**, and only if it overlaps that track's predicted box enough.
- Anything it can't link is dropped. `update_with_detections` only returns boxes that got a track ID.

**Why that is bad for the ball:**
- Low confidence: the model is often only 0.1 to 0.35 sure about the ball, so many ball boxes can never start a track.
- Tiny and fast: a ball box is about 15 px across and can move more than its own width in one frame (40 ms). Two boxes of the same ball in consecutive frames may barely overlap, so matching fails even when a track exists.
- Players are the opposite: big boxes, slow movement relative to their size, high confidence. ByteTrack suits them.

**The numbers (frames 0 to 299 of the DFL clip, conf 0.1):**

| Approach | Frames with a ball |
|---|---|
| Ball sent through ByteTrack (old) | 122 |
| Ball taken straight from YOLO (new) | 261 |

More than half the frames where the detector saw the ball were being thrown away. Silently: no error, no warning.

**General lesson:** a library's default settings can quietly filter your data. `conf=0.1` looked like "keep everything above 0.1", but a later stage had its own threshold. Count what goes into and comes out of each stage of a pipeline, not just the final output.

**In this project:** `track_frame` now splits detections by class. People go through NMS and ByteTrack; the ball skips both and is picked straight from the detector (the most confident ball above `ball_conf`). The ball's own tracker will be a Kalman filter (Track 1 step 6).

## Concept (2026-09-25): filtering with True/False masks

A **mask** is an array of True/False values, one per row, that says which rows to keep.

```python
class_id = np.array([2, 2, 0, 3, 2])   # player, player, ball, referee, player
is_ball = class_id == 0                 # [False, False, True,  False, False]
~is_ball                              # [True,  True,  False, True,  True ]
class_id[is_ball]                       # [0]
```

- Comparing an array with one number (`== 0`) compares every element and gives back a mask. No loop: numpy does the loop internally, in fast compiled code. This is called vectorised code.
- `~` flips every value. For True/False arrays use `~`, `&`, `|`, not Python's `not`, `and`, `or`, which only work on a single True/False.
- Indexing with a mask keeps the rows where it is True.

**It works on a whole `sv.Detections` table too.** `detection_sv[is_ball]` keeps the matching rows in *every* array at once (boxes, confidences, class IDs, `.data`), so they stay lined up.

```python
balls = detection_sv[is_ball]
people = detection_sv[~is_ball]
balls = balls[balls.confidence >= self.ball_conf]   # same trick on confidence
```

To combine conditions, use brackets: `(conf >= 0.3) & (class_id == 2)`. The brackets matter, because `&` is applied before `>=`.

**In this project:** this replaced a `for` loop with `break` that scanned `tracks.class_id` one row at a time. Shorter, faster, and harder to get wrong.

## Concept (2026-09-25): three small numpy details that matter

**`np.argmax` gives a position, not a value.**
```python
conf = np.array([0.2, 0.7, 0.4])
np.argmax(conf)   # 1, the index of the largest
conf.max()        # 0.7, the value itself
```
The position is what you need, to fetch the matching box: `balls.xyxy[best]`. The old code took the first ball in the list, which is arbitrary, since list order says nothing about which ball is real. The most confident one is a rule with a reason.

**Views vs copies.** Slicing a numpy array often gives a *view*: a new name for the same memory, not new data.
```python
a = np.array([1, 2, 3, 4])
v = a[0:2]        # view
v[0] = 99
a                 # [99, 2, 3, 4], the original changed too
```
`self.last_ball_bbox` has to survive into later frames. If it were a view into this frame's array and something later changed that array in place, the saved box would change behind your back, with no error. `np.array(..., copy=True)` guarantees its own memory. The bug it prevents is rare but very hard to find, so the copy is cheap insurance.

**numpy numbers vs Python numbers.** `np.argmax` returns `np.int64`, and `balls.confidence[best]` is `np.float32`. `int(...)` and `float(...)` turn them into plain Python numbers. Mostly tidiness, but it avoids surprises later: for example, `json.dumps` refuses an `np.float32`.

## Concept (2026-09-25): adding a row to a `Detections` table without breaking it

A `sv.Detections` is really several **parallel arrays**: `xyxy`, `confidence`, `class_id`, `tracker_id`, plus every array inside `.data` (`class_name`, `interpolated`). Row 5 of each must describe the same object, and nothing enforces that when you edit the arrays by hand.

If one array ends up a row short, one of two things happens later: an error when the table is filtered, or, worse, no error and every label after that row shifted onto the wrong object.

**How `_append_ball` adds one row safely:**
1. Build a one-row version of each column. `np.array([bbox])` wraps the box in a list so its shape is `(1, 4)` (one row of four corners), not a flat `(4,)`.
2. Match the existing number type: `dtype=tracks.class_id.dtype`. Joining an int array with a float array quietly turns everything into floats.
3. Stick it on the bottom: `np.vstack` for the 2D box table (rows x 4), `np.concatenate` for the flat 1D arrays.
4. Pad *every* key in `.data` with a sensible filler: the real flag for `interpolated`, `"ball"` for `class_name`, an empty string for other text columns, zeros for numbers.

**Why ball ID `-1`.** ByteTrack's IDs start at 1. A value it can never produce (a *sentinel*) means the ball can never share an ID with a player. The old code used `1`, which collided with a real player.

**Order matters: create `interpolated` before appending.** The padding loop only walks keys that *already exist* in `.data`. So `tracks.data['interpolated'] = np.zeros(len(tracks), dtype=bool)` must run first (every person marked "not a guess"); then the loop sees the key and adds the ball's flag. If it ran after, that array would be one short, which is exactly the bug described above.

**Check it, don't trust it:** the step 3 test asserted `len(values) == len(tracks)` for every `.data` array on every frame. 300 frames, no failures.

## Lesson (2026-09-25): environment gotchas, and why versions get pinned

**A function call can't repeat an argument.** `model.predict(frame, imgsz=self.imgsz, ..., imgsz=self.imgsz)` is a `SyntaxError: keyword argument repeated`, and the whole file refuses to import, not just that line. Easy to do when adding an argument to a long call, so reread the full call after editing.

**`device="cpu"` makes GPU use a choice.** With no `device`, Ultralytics grabs any GPU it finds. Pinning `"cpu"`:
- keeps latency numbers comparable (every measurement so far was on the CPU);
- keeps the GTX 1650 idle unless a test deliberately asks for `"cuda"`.

Training stays on Colab regardless. A short inference run is a much lighter load than training.

**Two Pythons, two torches.** This machine has a global Python 3.14 and a project `.venv`. Same package versions, except torch: the global one is the CPU build, `.venv` has the CUDA build (`2.14.0+cu130`). The notebooks were running on the global one. Switching the notebook kernel to `.venv` would silently move YOLO onto the GPU, which is another reason for `device="cpu"`. VS Code's kernel picker (top right of a notebook) shows which interpreter is in use.

**Pin versions.** The smoke-test notebook ran `pip install supervision` with no version. `sv.ByteTrack` prints a deprecation warning and is **removed in supervision 0.31**, so the next reinstall would have pulled 0.31 and broken `Tracker`, with nothing in this repo changed. `requirements.txt` now lists exact versions (`supervision==0.30.4` and so on), and `pip install -r requirements.txt` rebuilds the same environment. Rule: a deprecation warning is a bug report with a date on it. Pin now, migrate on purpose later.

## Concept (2026-09-25): tuning frames vs held-out frames

Every setting chosen by looking at results (inference size, `ball_conf`, NMS mode, Kalman noise) is a small fit to the data it was chosen on. Measure accuracy on those same frames and the score comes out too good, because the settings were picked partly for how well they happen to do on exactly those frames.

**The split for the DFL clip (`08fd33_4.mp4`, 750 frames):**
- Frames 0 to 299: tuning. Look, change settings, rerun as often as needed.
- Frames 300 to 749: held back for the hand-label spot-check, scored only once the settings are frozen.

**The same idea shows up three times in this project:**
- Training: the Roboflow test split was scored once and must not be used to choose between model variants.
- Stage 1 now: tune on frames 0 to 299, grade on 300 to 749.
- Factors later: the locked holdout season/competition, untouched until every factor has passed on development data.

**Honest limits:** consecutive frames of one clip are nearly identical, so this split is far weaker than a different match. And frames 375 and 439 were already viewed on 2026-09-21. Held out here means "not tuned on", not "never seen". It stops the worst self-deception; it does not replace testing on other footage.

## Concept (2026-09-25): the Kalman filter, what it is and what it assumes

**The problem it solves.** You want to know where something is, but your measurements are noisy and sometimes missing. For the ball: the detector gives a slightly jittery box most frames, and nothing at all in about 13% of frames. A Kalman filter combines two sources of information to get a better estimate than either alone:
1. **What physics predicts**: if the ball was moving right at 10 px per frame, it is probably 10 px further right now.
2. **What the detector measured**: the box YOLO found this frame, if any.

**What it keeps track of.** Two things, every frame:
- The **state**: its best estimate of what it is tracking. Here `[x, y, vx, vy]`: position and velocity of the ball centre, in pixels and pixels per frame. Velocity is never measured directly; the filter works it out from how the position changes.
- The **uncertainty** (covariance matrix `P`): how unsure it is about each part of the state. This is what makes it more than a smoothing trick. The filter always knows how much to trust itself.

**The two-step loop, once per frame:**

1. **Predict** (always). Move the state forward with the motion model: `x = x + vx`, `y = y + vy`, velocity unchanged. Then *grow* the uncertainty, because the future is less certain than the present (the ball might have been kicked).
2. **Update** (only when there is a measurement). Compare the prediction with the detection and move the estimate part of the way towards the detection. How far depends on the **Kalman gain**: a ratio of "how unsure the prediction is" to "how unsure prediction plus measurement are together". Then *shrink* the uncertainty, because new information came in.

**A 1D worked example.** Predicted x = 100 px, prediction uncertainty sigma 5 px (variance 25). Detector says 110 px, measurement sigma 2 px (variance 4).
```
gain      K = 25 / (25 + 4) = 0.86        trust the detector 86%, the prediction 14%
estimate    = 100 + 0.86 * (110 - 100) = 108.6 px
new var     = (1 - 0.86) * 25 = 3.45      sigma 1.9 px: better than either input alone
```
If the prediction had been very sure (variance 1) and the detector sloppy (variance 25), the gain would be 0.04 and the estimate would barely move. The filter weighs each source by how reliable it is, automatically, every frame.

**When the ball is missed**, only step 1 runs. The estimate keeps moving along the last velocity and the uncertainty keeps growing. Measured on the DFL clip:

| Frames missed | 1 | 3 | 5 | 7 | 9 |
|---|---|---|---|---|---|
| Position sigma (px) | 4.8 | 15.8 | 30.7 | 48.5 | 68.8 |

That growing number is `ball_sigma`, drawn as the circle (radius 2 sigma, so the real ball should be inside about 95% of the time).

**The assumptions, and where the ball breaks them:**
- **Linear motion model.** It assumes constant velocity between frames. A ball is kicked, bounces, spins, and curves. The filter handles this by assuming unknown random accelerations (**process noise**, `Q`, here `accel_std = 4` px per frame squared). Bigger `Q` means "trust the motion model less, react faster to changes".
- **Gaussian (bell-curve) noise.** Errors are assumed to be small, symmetric and usually close to zero. A false detection 400 px away (a player's boot) is not Gaussian noise at all; it is a different object. Without protection, one such detection would drag the estimate across the pitch. That is what the gate is for (below).
- **Known, constant noise levels.** `Q` (motion noise) and `R` (measurement noise, here `meas_std = 2` px) are fixed numbers chosen by tuning. In reality a kick is a sudden huge acceleration and a blurry fast ball is measured worse than a still one.
- **Markov property.** The current state holds everything that matters; older history adds nothing. Fine for a ball.
- **The camera is fixed.** It isn't: broadcast cameras pan and zoom. In pixel coordinates, a camera pan looks like the ball accelerating. Stage 2 (homography to pitch coordinates in metres) removes that.

**The gate.** Before accepting a detection, the filter asks: is this detection plausibly where the ball could be? It measures the distance from the prediction in units of the prediction's own uncertainty (**Mahalanobis distance**). The same 30 px miss is suspicious one frame after a detection (predicted sigma about 5 px, so 30 px is about 6 sigma) and perfectly normal after 6 missed frames (sigma about 39 px, under 1 sigma). The cut-off, 9.21, is the 99% point of a chi-square distribution with 2 degrees of freedom (x and y): if the filter's model is right, the real ball falls inside the gate 99% of the time. On the tuning frames, the gate rejected a green player's boot that the detector called a ball with 0.52 confidence, 426 px from the prediction, while the filter kept following the real ball.

**Escape hatch.** If the filter misses for 3 frames in a row while something ball-like shows up outside the gate, it restarts there. The likely story is a kick it could not follow, not a false positive. Risk: a false ball that persists 3 or more frames while the real one is unseen can capture the filter.

**Forward-only, on purpose.** A Kalman *smoother* also runs backwards and uses future frames to improve past estimates; it looks better on recorded video. It is banned here: using future frames breaks point-in-time discipline and is impossible on a live stream. Every estimate uses only frames up to now.

**In this project:** `pipeline/detection/ball_filter.py` (`BallKalman`: `predict`, `update`, `distance_sq`, `position_std`), used by `Tracker._update_ball`. It replaced the carry-forward, which froze the ball where it was last seen. Median error 5 frames into a miss: 6.4 px with the filter, 22.5 px frozen.

## Concept (2026-09-25): tuning a filter when there is no ground truth

To choose Kalman settings you need to know which setting tracks the ball best, but nobody has labelled where the ball really is. The trick used here: **hide data you do have, and check whether the filter can predict it.**

1. From every frame where the ball was detected, copy the filter and run only `predict()` for k frames (k = 1, 3, 5, 9), as if the ball had gone missing.
2. Compare where it ends up with where the detector later saw the ball, using only confident detections (0.35 or higher) as the reference, since those are most likely the real ball.
3. Do the same for the old carry-forward (just keep the last position), as a baseline.

Median error in pixels on frames 0 to 299:

| Frames ahead | Kalman | Carry-forward |
|---|---|---|
| 1 | 1.4 | 4.6 |
| 5 | 6.4 | 22.5 |
| 9 | 13.8 | 40.1 |

**Reading a flat grid.** Fifteen combinations of process noise (1 to 16) and measurement noise (1 to 4) all scored within a couple of pixels of each other. When results are that flat, picking the single best cell is fitting noise: the "winner" would probably lose on a different clip. So the sensible defaults were kept. The useful conclusion is "Kalman beats carry-forward by about 3 times, whatever reasonable settings", not "accel 2, meas 2 is optimal".

**Caveats:** the reference is itself the detector, so the check measures agreement with confident detections, not truth. And it only used frames 0 to 299, keeping 300 to 749 clean for the hand-label check.

**Caching.** Running YOLO once, saving the ball detections to a file, and then replaying them through the filter made each setting take seconds instead of a minute. Separate the slow part (the detector) from the part you are experimenting with.

## Concept (2026-09-25): two-pass NMS, and choosing a threshold from a gap

**What was going on.** NMS as used before only compared boxes of the same class. The detector often put two boxes on the referee: one labelled `referee`, one labelled `player`. Different classes, so NMS never compared them, and both survived. That inflates the player count and can give one person two IDs.

**Why not just make NMS ignore class (at 0.3)?** All 24 cross-class overlapping pairs on the tuning frames were checked by eye:
- 21 were one referee boxed twice, overlap (IoU) 0.76 to 0.97.
- 3 were two different people, a referee standing right in front of a player, IoU 0.30 to 0.56.

Class-agnostic NMS at 0.3 would have deleted a real person in those 3 frames.

**The fix: two passes.**
1. Same-class NMS at 0.3, as before.
2. A second pass that ignores class but only removes boxes overlapping 0.7 or more: near-identical boxes, which is what a double-boxed person looks like.

Result: 24 cross-class pairs down to 3, the 3 real ones.

**Why 0.7:** it sits in the gap between the highest "two people" overlap (0.56) and the lowest "one person" overlap (0.76). Looking at the actual cases before choosing is what made this possible; a threshold picked blind would have been a guess. It is still a tuned value from 24 pairs on one clip, so it is logged as such.

## Lesson (2026-09-25): measure latency per stage, and expect noise

**Time each stage, not just the total.** `Tracker.track_frame` now records `last_timings`, one `time.perf_counter()` reading between stages. At imgsz 640 on this CPU:

| Stage | Mean ms |
|---|---|
| YOLO detection | 35.9 |
| Split + NMS | 0.7 |
| ByteTrack | 6.1 |
| Ball + Kalman | 0.1 to 0.3 |

YOLO is about 84% of the time. So speeding up NMS or the Kalman code is pointless; only the detector (smaller input, smaller model, GPU, export formats like ONNX or OpenVINO) can bring the pipeline under 40 ms. The earlier guess that NMS + ByteTrack cost about 13 ms was about twice the real figure. Measure, don't estimate.

**Resolution trade-off.** 960 px found the ball in 5 points more frames than 640, for about 20 ms more per frame. 1280 was no better than 960. Detecting at higher resolution was not the fix for weak ball detection.

**Latency numbers are noisy.** The same code measured 44.6 ms mean on one run and 53.9 ms on the next. Laptops change clock speed with temperature and power, and background programs compete for the CPU. Rules:
- Always drop frame 0 (it includes loading the model, not steady-state cost).
- Report the median and the 95th percentile, not just the mean.
- Compare two settings in the same run, back to back, not across different days.
- Treat a difference of a few ms on this laptop as noise.

## Concept (2026-09-25): ID switches, and what shirt colour can and can't fix

**What an ID switch is.** Two players overlap (typically both chasing the ball), and when they separate their track IDs have swapped. Seen in the 300-frame tracker video. ByteTrack matches boxes by position and overlap only; while two boxes sit on top of each other, it cannot tell who is who.

**What the reference repo (`abdullahtarek/football_analysis`) does.** Its tracker is plain ByteTrack, the same as ours, so it has the same switches. Shirt colour only comes in afterwards, to label teams: it clusters each player's shirt colour and then **remembers the team per track ID** (`player_team_dict`). If two IDs swap, their team labels swap with them, and nothing ever corrects it. It even hardcodes `if player_id == 91: team_id = 1`, a manual patch for one mistake. (No license, so ideas only, no code.)

**What colour can fix:**
- Swaps between **opposing** players: white shirt vs green shirt is easy to tell apart.
- Not swaps between **teammates**: identical shirts. Shirt numbers are rarely readable in a wide broadcast shot.

**Why that is mostly fine for this project.** The planned factors are team-level: pressure, dominance, shape. A total or average over a team's players does not change if two teammates swap IDs. A swap across teams does change it, because a player's movement suddenly counts for the other side. So colour fixes the swaps that actually matter here. Per-player features (fatigue of one player) would need teammate identity, but those are deferred anyway because the camera shows only part of the pitch.

**A better design than the reference:**
1. Assign a team to every box, every frame, from the shirt colour (upper half of the box), compared with the two team colours learned at the start. Goalkeepers and referees already have their own classes.
2. Run a separate tracker per team, so a white player's ID can never jump to a green player.
3. Don't lock a team per ID. Keep a running vote per ID; if a track's colour suddenly flips team, flag it as a likely ID switch. That is a switch detector we don't currently have.

**Status:** not built. First measure how often switches happen and how many cross teams (part of the hand-label check), then decide whether to build it now or in Stage 3.